# VN-GAT on Colab TPU (TensorFlow)

**Runtime -> Change runtime type -> TPU** before running anything.

This is the TensorFlow branch. TF talks to Cloud TPUs through `TPUStrategy`
with no extra package, which avoids torch_xla's version-fragile Colab install.

The model, losses and metrics are a direct port of the PyTorch branch and are
verified against the same numerical properties: equivariance to ~1e-15 in
float64, the transposed rotation head recovering A^T for every A, chance levels
matching theory, and a perfect prediction giving zero geometric loss. The
parameter count matches the PyTorch model exactly (376,592 at defaults).


### 1. Check the TPU


In [ ]:
import tensorflow as tf
print("TF", tf.__version__)
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.config.experimental_connect_to_cluster(resolver)
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)
print("replicas:", strategy.num_replicas_in_sync)


### 2. Dependencies (TF is preinstalled on Colab)


In [ ]:
!pip install -q libigl trimesh pyyaml


### 3. Mount Drive

Checkpoints go to Drive so a disconnected runtime does not lose the run. Colab
disconnects far more readily than a batch job, so this matters more here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/vngat', exist_ok=True)


### 4. Get the code (TF branch)


In [ ]:
%cd /content
!git clone -b <TF_BRANCH_NAME> <YOUR_REPO_URL> vngat
%cd /content/vngat


### 5. Get the data

Copying Breaking Bad to Drive once and symlinking is usually faster across
sessions than re-downloading, but the first copy is slow.


In [ ]:
%cd /content/vngat
!mkdir -p data
!ln -sfn /content/drive/MyDrive/breaking_bad/everyday_compressed data/everyday_compressed
!ln -sfn /content/drive/MyDrive/breaking_bad/data_split           data/data_split
!python -m scripts.inspect_data --root data | head -20


### 6. Sanity checks


In [ ]:
!python -m pytest tests -q


### 7. Measure the padding cost FIRST

XLA compiles one program per distinct shape combination, so batches are padded
into buckets. This prints how many programs XLA will build and how much compute
the padding wastes.

**Read it before committing a session.** A median edge padding factor of 1.5x
means the TPU must be more than 1.5x faster than the GPU on this workload just
to break even -- and this model is small (376k parameters) and scatter-heavy,
which is not where TPUs are strong.


In [ ]:
!python -m scripts.train_tpu --report_buckets --root_dir data \
    --data_subsets everyday_compressed --split_source official


### 8. Train

`batch_size` is **per replica**; 8 replicas gives an effective batch of 8x it,
and the learning rate is scaled by the replica count in the trainer.

The first epochs are dominated by XLA compilation -- judge throughput from
epoch 5 onward, not epoch 0.

Watch the `[val ] tilt / twist` line: **tilt ~ 0 with twist ~ 90** means the
axis is learned but the azimuth is not recoverable, which is structural and no
amount of training moves it. **tilt ~ 90** means there is still headroom.


In [ ]:
!python -m scripts.train_tpu \
    --config configs/colab_tpu.yaml --root_dir data \
    --checkpoint_dir /content/drive/MyDrive/vngat/tpu1 --tag tpu1 --resume none \
    --data_subsets everyday_compressed --split_source official \
    --batch_size 8 --steps_per_epoch 40 --val_steps 8 \
    --epochs 300 --lr 5e-4 --lr_min 5e-5 --num_workers 4


### 9. Resume a later session

`--epochs` is a TOTAL, not an increment -- the trainer prints how many epochs
actually remain, so a mistake here is visible rather than silent.


In [ ]:
!python -m scripts.train_tpu \
    --config configs/colab_tpu.yaml --root_dir data \
    --checkpoint_dir /content/drive/MyDrive/vngat/tpu1 --tag tpu1 --resume auto \
    --epochs 600 --lr 2e-4


### 10. Curves


In [ ]:
!python -m scripts.plot_history --checkpoint_dir /content/drive/MyDrive/vngat/tpu1 \
    --out /content/curves.png
from IPython.display import Image; Image("/content/curves.png")
